# Phase 0 — 교차모델 자기보고 도달성 정찰 (Reachability, cross-model)

**질문:** α=0(주입 없음)에서 **프롬프트 페르소나 유도**만으로 Likert 자기보고 digit이 움직이나? 그리고 neutral 자기보고가 ~3.0으로 재현되나(계측 sanity)?

지금까지 이 현상은 **gemma-2-9b-it 한 모델**에서만 봤다(N=1). 여기서는 **Mistral·Qwen·Gemma** 세 모델에 같은 절차를 돌려 **도달성·성격 착시가 모델에 따라 다른지** 정찰한다.

- **벡터/추출 불필요** — 모델만으로 실행(`phase0_reachability.py`).
- **범위:** 프롬프트 수준 도달성만. 주입 수준 이중해리(v_behavior/v_selfreport)는 별도의 무거운 전체 파이프라인(향후 Phase 4)이며 이 노트북 밖.
- **모델:** `google/gemma-2-9b-it`(앵커·게이트) · `mistralai/Mistral-7B-Instruct-v0.3`(Apache) · `Qwen/Qwen2.5-7B-Instruct`(Apache).

**실행 순서:** GPU → 의존성 → HF 로그인 → 번들(`phase0_bundle.zip`) 업로드 → 모델 설정 → 스모크 → 전체 → 비교표/다운로드.

> **런타임:** 무료 T4(16GB)면 7~9B가 빡빡 → 설정 셀에서 `STEER_4BIT=1` 권장. Colab Pro L4/A100면 `0`. **-it/Instruct 모델 전용.**

## 1. GPU + 의존성 + HF 로그인

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q "transformers>=4.45" accelerate huggingface_hub bitsandbytes sentencepiece python-dotenv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# transformers>=4.45: Qwen2.5 + Gemma-2 동시 지원 (기존 노트북의 >=4.42에서 상향).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # gemma-2-9b-it 라이선스 수락 필요. Mistral/Qwen(Apache)은 로그인 없어도 됨.

## 2. 번들 업로드

`phase0_bundle.zip` 선택 — `common.py` + `steering/{phase0_reachability,steer_eval,gemma_common}.py` + `data/facets_en.json` (5파일, 아티팩트 불필요).

In [ ]:
from google.colab import files
import zipfile, os, subprocess
print('phase0_bundle.zip 를 선택하세요 ...')
up = files.upload()
name = next(iter(up))
os.makedirs('/content/asteer', exist_ok=True)
with zipfile.ZipFile(name) as z: z.extractall('/content/asteer')
%cd /content/asteer
print(subprocess.run(['find','.','-maxdepth','2','-type','f'], capture_output=True, text=True).stdout)

## 3. 모델 목록 + 설정 (여기만 바꾸면 됨)

α=0이라 **reachability(dE)는 층과 무관** — 층은 내부 diff 측정 위치일 뿐. 그래서 Mistral/Qwen은 auto mid-layer(`None`)로 충분하고, Gemma만 기존 결과 재현 위해 20으로 고정한다.

In [ ]:
import os
# 무료 T4(16GB)면 '1'(4-bit) 권장. Colab Pro L4/A100면 '0'.
os.environ['STEER_4BIT'] = '1'
# 주의: STEER_LAYER env 는 설정하지 않는다(설정하면 전 모델에 강제됨). 층은 아래 --layer 로 모델별 지정.
MODELS = [
    {'slug': 'gemma9b',   'id': 'google/gemma-2-9b-it',               'layer': '20'},   # 앵커(기존 재현)
    {'slug': 'mistral7b', 'id': 'mistralai/Mistral-7B-Instruct-v0.3', 'layer': None},   # auto mid-layer
    {'slug': 'qwen7b',    'id': 'Qwen/Qwen2.5-7B-Instruct',           'layer': None},
]
print('STEER_4BIT =', os.environ['STEER_4BIT'])
for m in MODELS: print(' -', m['id'], '| layer', m['layer'] or 'auto')

## 4. 스모크 (모델별 ~1-2분, neutral E_ft≈3.0 이어야 계측 sanity)

In [ ]:
# persona+neutral, first_token 만, 생성답 생략. 모델 로딩 + 4-bit 적재 + 계측 sanity 확인용.
for m in MODELS:
    lay = f"--layer {m['layer']}" if m['layer'] else ""
    print(f"\n===== SMOKE {m['id']} =====")
    !python steering/phase0_reachability.py --model {m['id']} {lay} --smoke --out artifacts/vectors/phase0_{m['slug']}_smoke.json

## 5. 전체 실행 (persona/grounded/fewshot × first_token+completion + 내부측정)

In [ ]:
for m in MODELS:
    lay = f"--layer {m['layer']}" if m['layer'] else ""
    print(f"\n===== FULL {m['id']} =====")
    !python steering/phase0_reachability.py --model {m['id']} {lay} --out artifacts/vectors/phase0_{m['slug']}.json
# (옵션) 내부 신호를 층별로 보려면 위 !python 줄 끝에 --layer-sweep 추가.

## 6. 교차모델 비교표 + 다운로드

In [ ]:
import json, os
from google.colab import files
hdr = f"{'model':10} {'L':>3} {'neutralE':>8} {'personaDE':>9} {'dN':>6} {'spec':>5} {'diag':>11}  rec"
print(hdr); print('-' * len(hdr))
for m in MODELS:
    p = f"artifacts/vectors/phase0_{m['slug']}.json"
    try:
        r = json.load(open(p))
    except FileNotFoundError:
        print(f"{m['slug']:10}  (missing {p})"); continue
    per = r['summary']['per_induction'].get('persona', {})
    print(f"{m['slug']:10} {str(r.get('layer')):>3} {str(r['neutral'].get('E_ft')):>8} "
          f"{str(per.get('dE_ft')):>9} {str(per.get('dN_ft')):>6} {str(per.get('spec_ratio')):>5} "
          f"{str(per.get('diagnosis')):>11}  {r['summary'].get('recommended_for_vector')}")
print('\n해석: neutralE≈3.0(계측 sanity) & personaDE 큼 → reachable(프롬프트로 digit 열림).')
print('      모델간 personaDE/diagnosis 가 다르면 → 도달성·성격 착시가 모델 의존적이라는 신호.')
for m in MODELS:
    p = f"artifacts/vectors/phase0_{m['slug']}.json"
    if os.path.exists(p): files.download(p)

---
### 해석 가이드
- **neutral E_ft ≈ 3.0** — 계측이 살아있다는 sanity. 크게 벗어나면 그 모델의 토크나이저/채점 배선 점검.
- **persona dE_ft 큼(reachable)** — 프롬프트 유도로 자기보고 digit이 열린다 = 그 모델에서도 "문은 안 잠김".
- **모델간 비교** — dE/진단이 비슷하면 현상이 강건(cross-model). 다르면 도달성·착시가 **모델 의존적**(그 자체가 발견).

### 이 정찰이 답하지 않는 것
- **주입 수준 이중해리**(v_behavior가 행동만 밀고 자기보고는 안 미는가, v_selfreport가 그 반대인가)는 **모델별 벡터 재추출**이 필요한 전체 파이프라인(extract→build→steer_eval, 향후 Phase 4). 이 노트북은 **프롬프트 수준 도달성**만.

### 참고 (모델별)
- **4-bit:** 7~9B를 T4에 얹으려면 `STEER_4BIT=1`(설정 셀). 첫 스모크에서 OOM이면 L4/A100로 올리거나 모델 수를 줄인다.
- **Qwen no-BOS:** Qwen은 BOS 토큰이 없어 `pooled_hidden`의 BOS-드롭이 어긋나지만, 그건 `extract_activations` 전용이고 phase0는 안 써서 **여기선 무관**.
- **attn eager:** `load_model`이 `attn_implementation="eager"`(Gemma softcapping 보존)라 Mistral/Qwen엔 다소 느리지만 결과엔 안전.